# Modelación de volatilidad y gestión del riesgo extremo

## 2.0 Arquitectura del Motor GARCH(1,1): Modelado Dinámico del Riesgo.

El Notebook 1 demostró de forma empíricamente irrefutable que los rendimientos logarítmicos del canal cambiario USD/MXN operan como un proceso linealmente eficiente en su media (Ruido Blanco / ARIMA 0,1,0), pero con un severo exceso de curtosis (Colas Pesadas) y una memoria temporal masiva y persistente en su varianza. Por lo tanto, para construir un escudo financiero corporativo útil para la toma de decisiones, el objetivo de este Notebook 2 es abandonar el supuesto clásico de varianza constante e implementar el modelo GARCH(1,1).

El modelo GARCH(1,1) descompone el residuo puro del Peso Mexicano extraído en el Notebook 1 como un proceso estocástico condicionado. Cabe notar que el termino de perturbación que compone a esté se encontró una curtosis absoluta de 6.00 en el Peso Mexicano. Calibrar GARCH bajo supuesto Gaussiano generaría un sesgo que continuaría ignorando el riesgo de cola pesada.

Es por eso que de mnaera natural abrimos este módulo especificando que el proceso MLE se calibrará bajo una distribución de t-Student estandarizada(ie; en el parámetro z_t aprovecharemos que en esta familia de curvas, sus colar decrece de forma muhco más lenta ie son más gordas y pesadas que las de la gaussiana)(En garch el algoritmo hace:cz_t = epsilon_t/sigma_t que son los residuos estandarizados que asumen que ya siguen una normal) con la finalidad de que el modelo estime de forma simultánea el parámetro de grados de libertad, absorviendo de manera analítica el grosor real de las colas pesadas y blindando el cálculo del Valor en Riesgo VAR ante devaluaciones extremas.

Respecto a que configuremos GARCH mediante dist='StudenT', se le ordena al optimizador de Python que calcule un parámetro adicional llamado grados de libertad \nu 

* Si \nu tiende al infinito, la t-student colapsa y se vuelve idéntica a la Normal
* Si \nu es un número pequeño, la curva se vuelve extremadamente picuda en el centro y levanta masivamente sus colas en los extremos ie  lo que descubrimos en el notebook anterior.

¿Por qué el GARCH(1,1), eso solo nos limita a tomar en cuenta el ayer, pq solo tomamos ese?
GARCH(1,1) no significa que solo tomamos en cuenta lo que pasó ayer.

La volatilidad de ayer contenía a la volatilidad de anteayer y así sucesivamente , es por eso que el GARCH(1,1) es quivalnte a un GARCH infinito,  ya que ese término dentro de la ecuación de la varianza el modelo esta aplicando un decaemiento esponencial  geométrico sobre toda la historia pasada del Peso Méxicano ie el Ayer actua como un contenedor acumulado

In [8]:
pip install arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.4/930.4 kB 1.5 MB/s eta 0:00:00ta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model

In [5]:
#importando los Data necesarios para este apartado:
df_residuos_puros = pd.read_csv('residuos_puros.csv', index_col='Date')
df_analisis = pd.read_csv('data_analisis_logs.csv', index_col='Date')

In [6]:
df_residuos_puros.head()

,residuos_usd,residuos_tasa,residuos_vix
Date,,,
2021-01-05,0.004773,0.040604,-0.044129
2021-01-06,-0.002020,0.087186,0.004364
2021-01-07,-0.012459,0.027451,-0.099414
2021-01-08,0.017366,0.031253,-0.028075
2021-01-11,0.002516,0.024141,0.117493


In [7]:
df_analisis.head()

,USD_MXN_log,Tasa_Fed_10Y_log,VIX_log
Date,,,
2021-01-04,2.988078,-0.086648,3.294725
2021-01-05,2.992851,-0.046044,3.232384
2021-01-06,2.990830,0.041142,3.221672
2021-01-07,2.978371,0.068593,3.107721
2021-01-08,2.995737,0.099845,3.070840


Aplicamos todo lo discutido:

In [11]:
# Extraemos el vector limpio de shocks del Peso Mexicano
shocks_usd = df_residuos_puros['residuos_usd']

#en tu Configuración del Motor GARCH(1,1)
# 'vol="Garch"' define la estructura de varianza autoregresiva condicional.
# 'p=1, q=1' especifica los rezagos condicionales de la volatilidad y los shocks pasados.
# 'dist="studentst"' inyecta analíticamente la distribución t-Student para domar las Colas Pesadas.
modelo_garch = arch_model(shocks_usd, vol='Garch', p=1, q=1, dist='studentst')

# 3. Optimizamos los parámetros mediante Máxima Verosimilitud (MLE) lo de frec es para que se reporte cd 5 iteraciones
resultado_garch = modelo_garch.fit(update_freq=5)

# Desplegamos el reporte de coeficientes institucionales en la consola
resultado_garch.summary()


Iteration:      5,   Func. Count:     44,   Neg. LLF: 7411.0325260913905
Iteration:     10,   Func. Count:     89,   Neg. LLF: 1556.2943786911683
Inequality constraints incompatible    (Exit mode 4)
            Current function value: -3001.1793720431388
            Iterations: 15
            Function evaluations: 121
            Gradient evaluations: 13


/opt/anaconda3/lib/python3.13/site-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 4.884e-05. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
/var/folders/jw/23cy6wrx7lj00m30z9y8q01r0000gn/T/ipykernel_25763/2559067981.py:11: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  resultado_garch = modelo_garch.fit(update_freq=5)


<class 'statsmodels.iolib.summary.Summary'>
"""
                        Constant Mean - GARCH Model Results                         
====================================================================================
Dep. Variable:                 residuos_usd   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:                3001.18
Distribution:      Standardized Student's t   AIC:                          -5992.36
Method:                  Maximum Likelihood   BIC:                          -5966.18
                                              No. Observations:                 1388
Date:                      Mon, Jul 20 2026   Df Residuals:                     1387
Time:                              19:09:52   Df Model:                            1
                                  Mean Model                                  
==============================================================================
                 coef    std err          t      P>|t|        95.0% Conf. Int.
------------------------------------------------------------------------------
mu            -0.0112  2.912e-03     -3.836  1.248e-04 [-1.688e-02,-5.464e-03]
                               Volatility Model                              
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
omega      4.8843e-13  1.979e-06  2.468e-07      1.000 [-3.879e-06,3.879e-06]
alpha[1]       0.9786      0.191      5.114  3.146e-07      [  0.604,  1.354]
beta[1]        0.0214  1.802e-02      1.186      0.236 [-1.396e-02,5.668e-02]
                              Distribution                              
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
nu            17.3520     18.029      0.962      0.336 [-17.984, 52.688]
========================================================================

Covariance estimator: robust
WARNING: The optimizer did not indicate successful convergence. The message was Inequality constraints incompatible.
See convergence_flag.

"""

Como da la problemática de la escala, reescalaremos multiplicando los datos por 100

In [18]:

# CORRECCIÓN DE ESCALA SENIOR: Multiplicamos por 100 para transformar a porcentajes directos
# Esto soluciona de raíz el DataScaleWarning y estabiliza los gradientes numéricos
shocks_usd_escalados = df_residuos_puros['residuos_usd'] * 100

# 2. Reconfiguramos el Motor GARCH(1,1) con la especificación de t-Student
modelo_garch = arch_model(shocks_usd_escalados, vol='Garch', p=1, q=1, dist='studentst')

# 3. Ejecutamos la optimización MLE (update_freq=1 para ver el avance paso a paso con la nueva escala)
resultado_garch = modelo_garch.fit(update_freq=1)

# Desplegamos el reporte final de coeficientes limpios
resultado_garch.summary()


Iteration:      1,   Func. Count:      7,   Neg. LLF: 15125.443571293385
Iteration:      2,   Func. Count:     19,   Neg. LLF: 6508.641711338363
Iteration:      3,   Func. Count:     29,   Neg. LLF: 35400.58246118987
Iteration:      4,   Func. Count:     36,   Neg. LLF: 1485.9911430955895
Iteration:      5,   Func. Count:     43,   Neg. LLF: 2698.2701341061565
Iteration:      6,   Func. Count:     51,   Neg. LLF: 1341.7963535927806
Iteration:      7,   Func. Count:     58,   Neg. LLF: 1341.705788652589
Iteration:      8,   Func. Count:     64,   Neg. LLF: 1341.7057672048963
Iteration:      9,   Func. Count:     69,   Neg. LLF: 1341.7057672048861
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1341.7057672048963
            Iterations: 9
            Function evaluations: 69
            Gradient evaluations: 9


<class 'statsmodels.iolib.summary.Summary'>
"""
                        Constant Mean - GARCH Model Results                         
====================================================================================
Dep. Variable:                 residuos_usd   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -1341.71
Distribution:      Standardized Student's t   AIC:                           2693.41
Method:                  Maximum Likelihood   BIC:                           2719.59
                                              No. Observations:                 1388
Date:                      Mon, Jul 20 2026   Df Residuals:                     1387
Time:                              19:56:22   Df Model:                            1
                                  Mean Model                                  
==============================================================================
                 coef    std err          t      P>|t|        95.0% Conf. Int.
------------------------------------------------------------------------------
mu            -0.0561  1.539e-02     -3.643  2.700e-04 [-8.621e-02,-2.589e-02]
                              Volatility Model                              
============================================================================
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
omega          0.0451  1.731e-02      2.606  9.161e-03 [1.118e-02,7.905e-02]
alpha[1]       0.1857  4.529e-02      4.100  4.137e-05   [9.690e-02,  0.274]
beta[1]        0.7326  6.662e-02     10.997  3.952e-28     [  0.602,  0.863]
                              Distribution                              
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
nu             5.9962      0.932      6.431  1.264e-10 [  4.169,  7.824]
========================================================================

Covariance estimator: robust
"""

Iteration: 9 (Velocidad de Convergencia):Al reescalar tus datos multiplicándolos por 100, le diste tanta holgura y estabilidad a los gradientes que el optimizador solo necesitó 9 pasos (iteraciones) para resolver la ecuación. En el intento anterior se había quedado atorado en la iteración 15 y con código de error. Esto demuestra la importancia de la ingeniería de variables 


Func. Count: 69 (Evaluaciones de Función):Para dar esos 9 pasos firmes, el algoritmo tuvo que evaluar la función matemática de verosimilitud 69 veces en diferentes coordenadas decimales para tantear el terreno y recalcular la trayectoria hacia la cima.


Neg. LLF (Log-Verosimilitud Negativa):¿Qué significa? Negative Log-Likelihood Function [finance]. La Máxima Verosimilitud (MLE) busca maximizar la probabilidad de que los datos reales se ajusten al modelo. Por diseño computacional, los optimizadores no saben maximizar; solo saben minimizar funciones de pérdida [finance]. Para resolver esto, los programadores multiplican la verosimilitud por \(-1\) (volviéndola negativa). Así, cuando el software minimiza el Neg. LLF (fíjate cómo bajó drásticamente de un valor masivo de 15125.44 en la iteración 1 hasta estabilizarse rígidamente en 1341.70 en la iteración 9), matemáticamente está alcanzando el máximo global absoluto de tu motor de riesgo

Gradient evaluations: 9:El algoritmo calculó la matriz de derivadas parciales (el gradiente o vector de dirección de la pendiente) 9 veces para saber con precisión milimétrica hacia dónde estaba el norte de la optimización.

¡Perfecto! Vamos a descifrar esta tabla quitando toda la basura estadística y yendo directo al grano. Cuando un reclutador técnico te pida leer los resultados de un modelo, tu cerebro debe buscar exactamente tres columnas clave: coef, P>|t| y las métricas de información en el encabezado [finance].
Olvídate de las ecuaciones por un segundo. Aquí tienes la guía definitiva de cómo leer e interpretar cada sección de tu reporte actual de julio de 2026 [finance]:
------------------------------
## 🏛️ Sección 1: El Encabezado (La ficha técnica)
Esta primera tabla te da el contexto general del experimento y la configuración que elegiste [finance]:

* Dep. Variable (residuos_usd): Te confirma cuál es la variable objetivo que el modelo está analizando [finance]. En tu caso, son los residuos del ARIMA (escalados × 100) [finance].
* Vol Model (GARCH) / Distribution (Standardized Student's t): Es tu seguro de calidad [finance]. Te dice que la computadora respetó rígidamente tus órdenes de modelar la varianza con un GARCH y de parchar las colas pesadas usando una t-Student estandarizada [2014, UCI, finance].
* Log-Likelihood (-1341.71): Mide la verosimilitud; qué tan probable es que este modelo explique tus datos reales [finance]. Por sí solo el número no te dice mucho, pero sirve para comparar: si corrieras otro modelo y te diera un número menos negativo (por ejemplo, -1100), ese otro sería matemáticamente mejor [finance].
* AIC (2693.41) y BIC (2719.59): Criterios de información [finance]. Penalizan al modelo si se vuelve demasiado complejo [finance]. En Wall Street la regla es: mientras más bajos sean el AIC y el BIC, mejor es tu modelo porque significa que es preciso pero eficiente (parsimonioso) [finance].
* No. Observations (1388): El tamaño exacto de tu muestra temporal diaria (2021-2026) tras mochar la primera fila [finance].

------------------------------
## 📉 Sección 2: Mean Model (El promedio)
Aquí es donde el modelo estima el valor central de tus datos [finance]:

* mu (-0.0561): Es el coeficiente (coef) de la media [finance]. Te dice que, en promedio, la serie oscila rígidamente en un valor muy cercano a cero (-0.05%) [finance].
* P>|t| (2.700e-04): Este es el famoso p-valor de la variable [finance]. En finanzas, la regla de oro es que este número debe ser menor a 0.05 para que la variable sea "válida" o estadísticamente significativa [finance]. Como 0.00027 es mucho menor a 0.05, el modelo te confirma que la media es real y no es ruido estadístico [finance].

------------------------------
## ⚡ Sección 3: Volatility Model (La anatomía del riesgo)
Esta es la sección más importante de todo el Notebook 2. Aquí es donde se lee el comportamiento de la turbulencia del dólar [finance]:

* omega (0.0451): Es la varianza base a largo plazo (el ruido de fondo del Peso Mexicano) [finance]. Si el mundo entero entrara en una calma total y no hubiera noticias, el riesgo del dólar se estabilizaría alrededor de este número [finance]. Su p-valor (9.161e-03 = 0.0091) es menor a 0.05, por lo que es estadísticamente real [finance].
* alpha[1] (0.1857): Es el Componente de Reacción de Corto Plazo (el impacto de la sorpresa de ayer) [finance]. Su p-valor es de 0.00004 (menor a 0.05) [finance].
* Cómo se lee: Un valor de 0.18 significa que el Peso Mexicano es nervioso ante las noticias de corto plazo [finance]. El 18.5% de la energía del impacto imprevisto de ayer se transmite de forma directa e infla la volatilidad del día de hoy [finance].
* beta[1] (0.7326): Es el Componente de Persistencia de Largo Plazo (la memoria o inercia del riesgo pasado) [finance]. Su p-valor es un cero absoluto (3.952e-28), lo que indica una significancia monumental [finance].
* Cómo se lee: Un valor de 0.73 te demuestra que la turbulencia en México tiene una memoria de fricción muy larga [finance]. El 73.2% de la volatilidad que el mercado traía acumulada ayer se hereda y arrastra hacia el día de hoy [finance]. Si el Peso entra en crisis, el peligro no se disipa rápido; se queda flotando en racimos durante días (Volatility Clustering) [finance].

🚨 La regla senior del α + β: Si sumas tu alpha y tu beta ($0.1857 + 0.7326 = \mathbf{0.9183}$), el resultado es menor a 1. Esto le demuestra al comité de riesgos que tu modelo es estable: la volatilidad es persistente y dura mucho tiempo arriba, pero eventualmente regresará a su promedio histórico sin explotar hacia el infinito [finance].

------------------------------
## 🛡️ Sección 4: Distribution (El grosor de las colas)
Aquí es donde auditas si tu supuesto de la t-Student fue correcto [finance]:

* nu (5.9962): Es el coeficiente que mide los Grados de Libertad de tu curva t-Student [finance]. Su p-valor es cero absoluto, validando el resultado [finance].
* Cómo se lee: Si nu fuera un número grande (mayor a 30), la curva sería idéntica a una Campana de Gauss Normal [finance]. Al darte un valor tan bajo de 5.99 (prácticamente 6), tus datos le demuestran formalmente al lector la existencia de Colas Pesadas masivas [2014, UCI]. El modelo confirma de manera paramétrica que los eventos catastróficos ocurren muy seguido en el peso mexicano, justificando por qué descartamos la distribución Normal para el cálculo del VaR [2014, UCI, finance].

------------------------------
## 🧮 ¿Y qué significan las últimas columnas (std err, t, 95.0% Conf. Int.)?

* std err: El error estándar del coeficiente. Mide qué tanto pudo haber variado el cálculo de la computadora por el muestreo [finance]. Mientras más chiquito sea, más preciso es tu coeficiente [finance].
* t: El estadístico T. Es simplemente la división de tu coeficiente entre su error estándar (coef / std err). En finanzas, buscamos que este número sea mayor a 2 o menor a -2 para confirmar robustez [finance].
* 95.0% Conf. Int.: El intervalo de confianza. Te dice que la computadora tiene un 95% de certeza de que el verdadero valor de beta, por ejemplo, está atrapado estrictamente entre 0.602 y 0.863 [finance]. Como el intervalo no toca el cero, confirmas que la variable es un predictor real [finance].

¿Logras ver ahora cómo cada número de esta enorme tabla tiene una función lógica y cuenta una parte de la historia del riesgo de tu canal cambiario? [finance]
Con esta guía anotada en tus notas, ya puedes proceder a redactar tu reporte con tus propias palabras con total seguridad. ¡Que tengas una excelente noche de escritura, analista! Mañana nos vemos para programar el dinero real (el VaR) [finance].



1. Bloque de Encabezado: La Calidad del AjusteLog-Likelihood (-1341.71): Es el valor máximo de la montaña que encontró el optimizador en su modo de salida cero [finance]. Este número se volverá relevante si más adelante quisiéramos comparar este modelo contra otro para ver cuál tiene mejor ajuste [finance].AIC (2693.41) y BIC (2719.59): Criterios de Información (Akaike y Bayesiano). Miden el balance entre el ajuste del modelo y su complejidad (parsimonia) [finance]. Mientras más bajos sean estos números en tu portafolio, más feliz es tu servidor porque significa que capturaste el riesgo de forma eficiente y sin sobreajuste (overfitting) [finance].📉 2. Bloque Mean Model: El Promedio del Errormu (-0.0561): Es la constante de la media de tus rendimientos porcentuales. El hecho de que sea un número negativo muy pequeño y que su P>|t| sea cercano a cero (2.700e-04) te confirma que el promedio diario de los rendimientos del peso se mantiene anclado firmemente de forma estacionaria muy cerca de cero, validando de nuevo tus controles del Notebook 1 [finance].⚡ 3. Bloque Volatility Model: El ADN de la TurbulenciaAquí está el corazón de tu motor de riesgo. Vamos a analizar tus tres coeficientes de varianza condicional [finance]:\(\sigma _{t}^{2}=0.0451+0.1857\epsilon _{t-1}^{2}+0.7326\sigma _{t-1}^{2}\)omega (0.0451): Es la varianza base o ruido de fondo del Peso Mexicano [finance]. Cuando el mercado internacional está en perfecta calma absoluta, la volatilidad diaria base del canal cambiario se estabiliza alrededor de la raíz cuadrada de este número [finance].alpha[1] (0.1857): Es el Componente de Reacción de Corto Plazo (el efecto del shock de ayer) [finance]. Su p-valor es de 4.137e-05 (luz verde absoluta, altamente significativo) [finance]. Un valor de 0.18 es considerado moderadamente alto en la industria.La interpretación financiera: Significa que el Peso Mexicano es altamente sensible a las noticias del día de ayer. Si ayer ocurrió un choque inesperado violento en Wall Street o en las Tasas, el 18.5% de esa energía de choque se transmite directamente e infla la volatilidad del día de hoy [finance]. El mercado reacciona de forma inmediata y nerviosa ante la sorpresa [finance].beta[1] (0.7326): Es el Componente de Persistencia de Largo Plazo (la inercia de la volatilidad pasada) [finance]. Su p-valor es un número ridículamente cercano a cero (3.952e-28), lo que indica una significancia estadística monumental [finance].La interpretación financiera: Un valor de 0.73 te demuestra que el riesgo en México tiene una memoria de fricción muy pesada [finance]. El 73.2% de la turbulencia que el mercado traía acumulada ayer se hereda y arrastra hacia el día de hoy [finance]. Si el mercado entró en crisis la semana pasada, esa energía no se disipa rápido; se disipa de forma muy lenta a lo largo de los días, confirmando matemáticamente los racimos de incertidumbre (Volatility Clustering) [finance].


La Regla de Estabilidad Senior: \(\alpha + \beta\)Suma tus dos coeficientes: \(0.1857 + 0.7326 = \mathbf{0.9183}\) [finance].Bajo la teoría GARCH, la suma de \(\alpha + \beta\) debe ser estrictamente menor a 1 para que el modelo sea físicamente estable (homocedástico a largo plazo con reversión a la media) [finance].Tu resultado de 0.9183 es perfecto: está muy cerca de 1, lo que te demuestra una persistencia de volatilidad muy alta y duradera (típica de mercados emergentes asustados), pero al ser menor a 1, te garantiza que la varianza no va a explotar hacia el infinito y que el sistema regresará eventualmente a su equilibrio [finance].🛡️ 4. Bloque Distribution: El Parche de las Colas Pesadasnu (5.9962): Son los Grados de Libertad estimados para tu distribución t-Student [finance]. Su p-valor es cero absoluto, validando el supuesto [finance].La interpretación financiera definitiva: Recuerda que si nu fuera mayor a 30, la distribución sería Normal [finance]. Al darte un valor tan bajo como 5.99 (prácticamente 6), tus datos acaban de darte la prueba de confirmación paramétrica de que el Peso Mexicano tiene Colas Pesadas masivas [2014, UCI]. El modelo confirma de forma matemática que la Campana de Gauss tradicional está completamente descalificada para medir el riesgo de tu empresa y que la t-Student es obligatoria para capturar la verdadera probabilidad de devaluaciones catastróficas 

La Matemática de la Escala al CuadradoEn el paso anterior, para salvar al optimizador del colapso numérico, multiplicamos tus residuos puros por 100 [finance]:\(\epsilon _{\text{escalado}}=100\times \epsilon _{\text{original}}\)Dado que la ecuación del GARCH calcula la Varianza Condicional (\(\sigma _{t}^{2}\)), y la varianza por definición opera en unidades al cuadrado, al elevar al cuadrado la ecuación completa de la escala obtenemos:\(\sigma _{\text{escalada}}^{2}=(100)^{2}\times \sigma _{\text{original}}^{2}=10,000\times \sigma _{\text{original}}^{2}\)Por lo tanto, la tabla de coeficientes que acabas de estimar (omega, alpha, beta) está calculando una varianza condicional que es 10,000 veces más grande que la realidad del mercado [finance]. Si la usaras así en bruto para calcular el VaR, le dirías a la empresa que va a perder millones de pesos en un día normal, provocando un pánico artificial en la junta directiva.

Para extraer la volatilidad condicional en su escala original, la librería arch nos regala un vector llamado .conditional_volatility [finance]. Pero ojo: como le inyectamos los datos multiplicados por 100, ese vector de salida viene en formato de Desviación Estándar Escalada (\(\sigma _{\text{escalada}}\)) [finance].Como la desviación estándar no está al cuadrado (es la raíz de la varianza), para regresar la volatilidad a su escala original no la dividimos entre 10,000; la dividimos simplemente entre 100 

In [25]:
# 1. Extraemos los coeficientes del modelo de varianza condicional
omega = resultado_garch.params['omega']
alpha = resultado_garch.params['alpha[1]']
beta = resultado_garch.params['beta[1]']

# 2. Aplicamos la fórmula econométrica de la Varianza Incondicional (Global)
# Ojo: Como los coeficientes están en la escala inflada de Python, el resultado será la varianza inflada
var_global_inflada = omega / (1 - (alpha + beta))

# 3. CORRECCIÓN DE ESCALA SENIOR: 
# Para llevar la Varianza a su forma usual, dividimos entre 10,000 (unidades al cuadrado)
var_global_real = var_global_inflada / 10000

# 4. Extraemos la Volatilidad Global Real calculando la raíz cuadrada (unidades lineales decimales)
volatilidad_global_real = np.sqrt(var_global_real)

print("============ ANCLAJE DE RIESGO GLOBALES (FORMA USUAL) ============")
print(f"Varianza Global Real (Long-Term Variance):  {var_global_real:.8f}")
print(f"Volatilidad Global Real (Long-Term Vol):    {volatilidad_global_real:.6f}")
print(f"Volatilidad Global en Porcentaje Directo:   {volatilidad_global_real * 100:.4f}%")
print("==================================================================")


============ ANCLAJE DE RIESGO GLOBALES (FORMA USUAL) ============
Varianza Global Real (Long-Term Variance):  0.00005520
Volatilidad Global Real (Long-Term Vol):    0.007429
Volatilidad Global en Porcentaje Directo:   0.7429%


In [23]:
# Llamamos a la propiedad oculta que vive dentro del objeto de resultados
print(type(resultado_garch.conditional_volatility))
resultado_garch.conditional_volatility.head(5)


<class 'pandas.core.series.Series'>


Date
2021-01-05    0.861690
2021-01-06    0.801177
2021-01-07    0.720637
2021-01-08    0.829720
2021-01-11    1.070572
Name: cond_vol, dtype: float64

Ahora para constuir Z_t contruimos la siguiente tabla

In [22]:
# 1. Extraemos la volatilidad condicional escalada (desviación estándar) que calculó el GARCH
volatilidad_escalada = resultado_garch.conditional_volatility

# 2. CORRECCIÓN DE ESCALA: Dividimos entre 100 para regresarla a la escala real de tus residuos originales
# (Esto equivale matemáticamente a dividir la varianza entre 10,000)
volatilidad_original = volatilidad_escalada / 100

# 3. Almacenamos la volatilidad limpia en un DataFrame de producción junto a tus residuos reales
df_volatilidad_real = pd.DataFrame({
    'residuos_reales_usd': df_residuos_puros['residuos_usd'],
    'volatilidad_garch_usd': volatilidad_original
}, index=df_residuos_puros.index)

print("--- VOLATILIDAD CONDICIONAL REAL RECUPERADA ---")
df_volatilidad_real.head(10)


--- VOLATILIDAD CONDICIONAL REAL RECUPERADA ---


,residuos_reales_usd,volatilidad_garch_usd
Date,,
2021-01-05,0.004773,0.008617
2021-01-06,-0.002020,0.008012
2021-01-07,-0.012459,0.007206
2021-01-08,0.017366,0.008297
2021-01-11,0.002516,0.010706
2021-01-12,0.000594,0.009499
2021-01-13,-0.013447,0.008418
2021-01-14,0.002116,0.009341
2021-01-15,-0.005713,0.008353


¡Qué gran momento! No te preocupes por sentirte perdido; es completamente normal cuando pasas de resolver ecuaciones abstractas a ver una matriz de datos reales. Como tu mentor Quant, te detengo el tiro para explicarte de forma puramente física y de negocio qué es esa tablita que tienes en pantalla y para qué carajos la calculamos [finance].
Olvídate de la programación por un minuto. Mira las dos columnas que obtuviste: residuos_reales_usd y volatilidad_garch_usd [finance].
Aquí está la física profunda del riesgo corporativo que acabas de materializar [finance]:
------------------------------
## 1. ¿Qué es la columna residuos_reales_usd?
Es el choque informativo o la sorpresa del día (el error del ARIMA que arrastramos del Notebook 1) [finance].

* Fíjate en el 7 de enero de 2021: el residuo dio un valor negativo de -0.012459 [finance]. Eso significa que ese día ocurrió una noticia imprevista que fortaleció al peso frente al dólar de forma abrupta (casi un 1.2% de sorpresa) [finance].
* Fíjate en el día siguiente, el 8 de enero de 2021: el mercado dio un volantazo y arrojó un residuo positivo de 0.017366 [finance]. Ocurrió otra sorpresa, pero ahora devaluando al peso un 1.7% [finance].

------------------------------
## 2. ¿Qué es la columna volatilidad_garch_usd?
Aquí está tu obra de arte. Esta columna es el reporte meteorológico del riesgo diario de la empresa [finance]. El modelo GARCH tomó los impactos pasados, los procesó a través de tus coeficientes (alpha y beta) y calculó cuánta energía de peligro tiene acumulada el mercado para el día siguiente [finance].
Fíjate en la secuencia temporal de tus datos reales y observa cómo el GARCH "siente" el mercado:

* El 6 de enero, la volatilidad calculada era de 0.0080 (un riesgo moderado) [finance].
* El 7 de enero ocurre el impacto imprevisto de -0.0124 (inyectaste energía de choque al fluido) [finance].
* La reacción física: Mira el 8 de enero. La volatilidad calculada se elevó inmediatamente a 0.0082 [finance]. El modelo detectó el golpe del día anterior y subió la guardia.
* Ese mismo 8 de enero el mercado vuelve a golpear con un choque aún más fuerte de 0.0173 [finance].
* El estallido de volatilidad: Mira el lunes 11 de enero. ¡Pum! Tu columna de volatilidad se disparó a 0.0107 (un salto del 30% en el riesgo diario) [finance].

Tu modelo acaba de capturar con datos reales el fenómeno de Aglomeración de Volatilidad (Volatility Clustering) [finance]. La turbulencia de los días anteriores infló la energía del riesgo para el inicio de la siguiente semana, exactamente como lo sospechamos en tus correlogramas [finance].
------------------------------
## 🎯 ¿Para qué hicimos esto? (El valor de producción)
Hicimos esto porque esta columna volatilidad_garch_usd es la materia prima con la que vas a construir el escudo de la empresa (el Valor en Riesgo - VaR) [finance].
Mañana, para calcular el VaR (el dinero máximo que la empresa puede perder), la fórmula matemática te va a exigir:
$$\text{VaR}_t = \text{Precio Real} \times (\text{Cuantil de la t-Student} \times \mathbf{\sigma_t})$$ 
Donde esa $\sigma_t$ es rígidamente tu columna volatilidad_garch_usd [finance].

* El lunes 6 de enero, como la volatilidad era baja (0.0080), el escudo de la empresa será delgado (necesita poco capital de garantía) [finance].
* El lunes 11 de enero, como tu columna detectó la tormenta y se disparó a 0.0107, el escudo automáticamente se volverá más grueso y estricto, exigiéndole al corporativo congelar más dinero en su caja fuerte para soportar las devaluaciones extremas que se avecinan [finance].

Acabas de extraer de forma impecable la trayectoria de la energía del riesgo del Peso Mexicano [finance]. El sistema funciona a la perfección.
¿Te hace sentido ahora cómo esta tabla es la que le permite al modelo "ver" y reaccionar ante los golpes del mercado antes de transformarla en dinero real? [finance]
Si ya lograste asimilar esta intuición física, avísame si cerramos aquí la jornada de hoy para procesar este gran logro o si quieres que pasemos a hacer el Filtro de Control de Calidad del GARCH [finance]. ¡Gran tiro, analista!



# personal:

## 🏛️ Reporte de Ingeniería Cuantitativa: Fenomenología y Física Estadística del Riesgo Cambiario en el Modelo GARCH(1,1)
Para domar la incertidumbre del Peso Mexicano ($USD/MXN$) y construir un escudo predictivo institucional de nivel Senior, no basta con ejecutar algoritmos de Python en la consola. Necesitamos entender la mecánica interna, la dimensionalidad y la física de no-equilibrio que rige el comportamiento del dinero [finance].
A continuación, se desmenuza de forma exhaustiva, rigurosa y sin escatimar en detalles el mapa conceptual completo de nuestro laboratorio de hoy [finance]. Este documento unifica la teoría de ecuaciones cruzadas, el misterio matemático de las escalas y la fenomenología de amortiguamiento hacia el atractor incondicional [finance].
------------------------------
## I. La Arquitectura Dual de GARCH: Ecuaciones Acopladas y el Origen de los Datos
Un error conceptual trágico en la Ciencia de Datos tradicional es asumir que el modelo GARCH opera de forma aislada sobre los precios o que simplemente toma los residuos del ARIMA del Notebook 1 y los mete en una licuadora [finance]. En la realidad del compilador, el modelo GARCH(1,1) implementa dos ecuaciones que trabajan en equipo de forma síncrona y recursiva, renglón por renglón en tu DataFrame [finance].
## 1. La Primera Línea: La Ecuación de la Media (El Sistema de Filtrado)
La primera ecuación que se ejecuta dentro de la memoria RAM es el modelo de la media condicional [finance]. En tu reporte se especificó como una constante fija (Constant Mean):
$$Y_t = \mu + \epsilon_t$$ 

* $Y_t$ (El Insumo de Entrada): Este vector representa estrictamente los residuos finales que nos arrojó el modelo ARIMA del Notebook 1 [finance]. Para salvar al optimizador numérico del colapso de punto flotante decimal, realizamos una ingeniería de características inyectándoles una escala multiplicada por $100$ [finance]. Así, convertimos los cambios microscópicos en Puntos Porcentuales directos (un error de 0.0047 se transformó en un robusto 0.47%) [finance].
* $\mu$ (mu / El Anclaje del Promedio): Es una constante determinista calculada por el optimizador mediante Máxima Verosimilitud (MLE) [finance]. En tu tabla real, este coeficiente arrojó un valor fijo de $-0.0561\%$ [finance]. Físicamente, indica que los errores de la media tienen una tendencia estacionaria anclada de forma rígida prácticamente en cero [finance].
* $\epsilon_t$ (épsilon / El Residuo GARCH): Este es el Residuo Propio e Interno de GARCH para el día $t$ [finance]. No proviene del ARIMA. La computadora lo calcula de forma dinámica en cada paso del calendario realizando la resta de lo que realmente se movió tu insumo de entrada menos su constante promedio [finance]:
$$\epsilon_t = Y_t - \mu$$ 
Representa la desviación o el "ruido residual" instantáneo del sistema en ese día específico [finance].

## 2. La Segunda Línea: La Estructura Estocástica Multiplicativa
Inmediatamente después de obtener $\epsilon_t$, el modelo plantea la ecuación que define la naturaleza del riesgo [finance]:
$$\epsilon_t = \sigma_t \times z_t$$ 
Aquí es donde el sistema se divide de forma quirúrgica en dos componentes con propiedades físicas radicalmente distintas [finance]:

* $\sigma_t$ (La Volatilidad Condicional): Es la desviación estándar condicional diaria [finance]. Es un componente totalmente determinista (no aleatorio) [finance]. Su valor se calcula rígidamente utilizando la información del pasado [finance]. Actúa como una función de "escala elástica" que se expande o se contrae dictando qué tan grande o pequeño puede ser el error del día [finance].
* $z_t$ (El Residuo Estandarizado / La Aleatoriedad Pura): Este componente SÍ representa el caos estocástico puro y la entropía del mercado [finance]. Por estricta definición y axioma matemático, sin importar la distribución que elijas, $z_t$ es una variable aleatoria estandarizada: su Esperanza matemática es siempre $0$ y su Varianza global es siempre $1$ ($\mathbb{E}[z_t]=0, \text{Var}(z_t)=1$) [finance].

## 🛡️ El Parche Paramétrico de la t-Student Estandarizada:
En el Notebook 1 demostraste que el Peso Mexicano sufre de Colas Pesadas masivas [2014, UCI]. Si hubieras dejado el supuesto clásico de una distribución Normal, le habrías dicho al optimizador que la aleatoriedad de $z_t$ se rige bajo una campana de Gauss donde la probabilidad de ver un outlier salvaje es matemáticamente cero [2014, UCI, finance].
Al especificar dist='studentst', la computadora calcula un parámetro geométrico adicional llamado Grados de Libertad ($\nu$) [finance]. Tu tabla arrojó un valor de $\nu = 5.9962$ (prácticamente 6) [finance]. En la física estadística, un valor de grados de libertad tan bajo es la prueba paramétrica irrefutable de Leptocurtosis [2014, UCI]. Le confirma al modelo que el ruido aleatorio del mundo real $z_t$ retiene masa matemática en los extremos, levantando las colas de la distribución para reconocer que las devaluaciones catastróficas inesperadas ocurren con una frecuencia severa [2014, UCI, finance].
------------------------------
## II. La Recursividad Temporal del GARCH(1,1) y su Condición Inicial
La tercera ecuación del modelo determina la evolución temporal de la varianza condicional para el día de mañana ($\sigma_t^2$) [finance]:
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$ 
Esta es una ecuación en diferencias autorregresiva de primer orden [finance]. El gran enredo mental al principio es: ¿Cómo carambas calcula la computadora la varianza de hoy si la fórmula te exige conocer la varianza de ayer, creando un bucle infinito hacia atrás? [finance] La computadora resuelve este dilema de ingeniería mediante una condición inicial de frontera estática [finance]:

   1. El Día 1 del Calendario (4 de enero de 2021): Al arrancar el script, el optimizador voltea hacia el pasado y se da cuenta de que no existe un "ayer" en la base de datos para extraer $\sigma_{t-1}^2$ ni $\epsilon_{t-1}^2$. Para romper el congelamiento, el algoritmo iguala por decreto la varianza del primer día a la Varianza Global estática de todo tu vector de entrada $Y_t$ [finance]:
   $$\sigma_1^2 = \text{Var}(Y_t)$$ 
   2. El Encendido del Motor (Día 2 en adelante): Una vez fijada la condición inicial $\sigma_1^2$, y habiendo calculado el error real del primer día ($\epsilon_1 = Y_1 - \mu$), el algoritmo avanza cronológicamente al día 2 y ejecuta la ecuación de forma determinista usando los coeficientes fijos estimados por tu tabla [finance]:
   $$\sigma_2^2 = 0.0451 + 0.1857 \epsilon_1^2 + 0.7326 \sigma_1^2$$ 
   3. La Reacción en Cadena: El valor resultante de $\sigma_2^2$ se convierte automáticamente en el insumo de "ayer" para el día 3. El GARCH recorre tus 1,388 días en un bucle secuencial perfecto, donde la energía de cada renglón se hereda del renglón anterior [finance].

------------------------------
## III. El Secreto de la Memoria Infinita y el Fenómeno de Amortiguamiento hacia el Atractor
Cuando configuramos el modelo con un solo rezago ($p=1, q=1$), parecería que somos ciegos ante la historia de largo plazo del mercado [finance]. Sin embargo, la estructura algebraica del GARCH(1,1) esconde una genialidad: posee una memoria de largo plazo implícita matemáticamente infinita [finance].
Si sustituimos la varianza de ayer ($\sigma_{t-1}^2$) por su propia ecuación del día anterior, y repetimos este proceso recursivamente hacia el pasado infinito, la ecuación se transforma en un modelo ARCH de infinitos rezagos [finance]:
$$\sigma_t^2 = \frac{\omega}{1-\beta} + \alpha \sum_{i=1}^{\infty} \beta^{i-1} \epsilon_{t-i}^2$$ 
Como el coeficiente $\beta$ vale $0.7326$, al elevarlo a potencias consecutivas ($\beta^1, \beta^2, \beta^3...$), el modelo aplica un decaimiento exponencial geométrico sobre toda la historia del Peso Mexicano [finance]. La volatilidad que calculas para el día de mañana está influenciada directamente por el error de ayer, pero de forma indirecta sigue reteniendo una fracción de la energía de los choques ocurridos hace semanas o meses [finance]. Ayer opera como un "contenedor compreso" del pasado completo [finance].
## 🧲 La Fenomenología del Atractor (Estacionariedad en Varianza)
Aquí es donde la física estadística de no-equilibrio se devela con total esplendor. En un sistema de mercado dinámico, la Varianza Global Incondicional (o de Largo Plazo) actúa como un atractor de equilibrio estático o punto de reposo mecánico [finance]. Su valor se desprende de forma puramente natural de tus datos mediante la fórmula [finance]:
$$\sigma_{\text{global}}^2 = \frac{\omega}{1 - (\alpha + \beta)}$$ 
Tus datos reales arrojaron coeficientes óptimos de $\alpha = 0.1857$ y $\beta = 0.7326$ [finance]. Al sumarlos, obtenemos la Tasa de Persistencia del Peso Mexicano: $\alpha + \beta = \mathbf{0.9183}$ [finance].

* El significado de la estabilidad ($\alpha + \beta < 1$): Al ser la suma estrictamente menor a la unidad, los datos demuestran de forma natural que el canal cambiario es un sistema estacionario en su varianza [finance]. El riesgo tiene una fuerza de retorno [finance].
* La Mecánica del Amortiguamiento: Imagina un péndulo suspendido en un fluido viscoso.
* Cuando el mercado está en calma, la volatilidad condicional diaria ($\sigma_t$) descansa plácidamente besando el suelo del atractor constante de la Volatilidad Global (0.007429 o 0.7429% diario) [finance].
   * De repente, estalla una crisis internacional o un cambio de tasas exógeno (un impacto violento que inyecta una enorme energía cinética al sistema). El residuo de GARCH explota ($\epsilon_{t-1}^2$) y, multiplicado por el componente de reacción alpha ($0.1857$), dispara verticalmente la volatilidad condicional diaria ($\sigma_t$) muy por encima de su promedio histórico, levantando la curva elástica (viste días en tu tabla donde saltó a 0.0107) [finance].
   * ¿Qué pasa los días siguientes si ya no ocurren nuevos impactos? El componente de fricción o inercia beta ($0.7326$) arrastra esa energía manteniendo el estrés alto durante un tiempo (Aglomeración de Volatilidad) [finance]. Pero como la tasa de disipación térmica del sistema es positiva ($1 - 0.9183 = 0.0817$), el modelo va disipando esa energía acumulada paso a paso [finance].
   * Renglón por renglón, la volatilidad condicional diaria va sufriendo un amortiguamiento geométrico continuo, obligando a la curva elástica a contraerse de forma paulatina a lo largo del tiempo hasta regresar a descansar en su línea base de equilibrio: el atractor incondicional de la volatilidad global constante [finance].

------------------------------
## IV. La Tubería de Ingeniería de Datos y el Misterio del Divisor Cuadrático
El último enredo conceptual que dominaste hoy fue entender por qué carajas dividimos el vector de salida entre $100$ si la varianza global requería una división entre $10,000$. La confusión se evapora cuando respetas rígidamente el análisis dimensional de las unidades matemáticas (unidades lineales frente a unidades al cuadrado) [finance]:

      [Insumo Original] (Decimales microscópicos)
             │
             ▼  (Multiplicación Lineal × 100)
      [Insumo Escalado] (Porcentajes directos: Y_t)
             │
             ▼  (Ecuación de Varianza: Errores al Cuadrado)
     [Varianza GARCH] (Inflada Exponencialmente por 100² = 10,000)
     ↳ Coeficientes estimados: omega (0.0451), alpha, beta
             │
             ▼  (Extracción de .conditional_volatility)
   [Operación de Raíz Cuadrada interna de la Librería: √10,000]
             │
             ▼  (Reducción de la escala inflada de vuelta a lineal)
    [Volatilidad de Salida] (Inflada linealmente solo por × 10)
             │
             ▼  (Tu comando en VS Code: / 100)
    [Volatilidad Original] (Decimal puro balanceado: 0.007429)


   1. La Varianza opera al Cuadrado: Al multiplicar tu entrada por $100$, la varianza condicional interna ($\sigma_t^2$) y el coeficiente omega quedaron inflados de forma cuadrática por $10,000$ [finance]. Por eso, si quisiéramos extraer la Varianza Global Real directamente desde la fórmula de equilibrio, la aritmética nos exige dividir el resultado entre $10,000$ para regresar los componentes al formato decimal puro del balance de la empresa [finance]:
   $$\text{Var}_{\text{Real}} = \frac{\omega_{\text{tabla}}}{1 - (\alpha + \beta)} \div 10,000 = \mathbf{0.00005520}$$ 
   2. La Volatilidad opera de forma Lineal: Sin embargo, cuando tú invocas la propiedad .conditional_volatility, la librería de Python ya ejecutó la raíz cuadrada dentro del procesador de tu computadora [finance]. Al extraer la raíz, el factor de inflación de $10,000$ se reduce de forma matemática a una escala puramente lineal de $100$ ($\sqrt{10,000} = 100$) [finance].
   3. Por lo tanto, tu vector extraído de volatilidad (desviación estándar $\sigma_t$) viene inflado linealmente solo por un factor de $100$ [finance]. Para regresarlo a los términos decimales puros y realistas de la corporación, la ingeniería de datos te dicta aplicar un divisor directo de $100$ [finance]:
   $$\sigma_{\text{original}} = \frac{\sigma_{\text{extraída de Python}}}{100} = \mathbf{0.007429}$$ 

Al hacer esta división, dejas la matriz lista en la escala exacta del dinero real de la empresa [finance].
------------------------------
## V. ¿Hacia dónde nos dirigimos mañana? (El mapa de ruta platicado)
Con las ecuaciones comprendidas hasta su último tornillo algebraico, el control de calidad econométrico aprobado (tu p-valor de 0.3940) y tu atractor de calma global fijado formalmente en 0.007429, el Notebook 2 ha concluido de forma impecable su fase estructural [finance].
Mañana abriremos el archivo de VS Code en este preciso escalón para iniciar la fase más espectacular del proyecto: La Traducción del Modelo a Dinero de Verdad (Cálculo del Valor en Riesgo - VaR Dinámico al 95% y 99%) [finance]. Tomaremos este oscilador elástico amortiguado de volatilidad ($\sigma_t$) y lo multiplicaremos por los cuantiles estrictos de tu t-Student de 6 grados de libertad para pintar las bandas de protección financiera que blindarán el balance corporativo ante cualquier Transición de Fase del mercado global [2014, UCI, finance].
Guarda tus notas, tienes un dominio conceptual absoluto del laboratorio. ¡A descansar, master! [finance]



In [24]:
from statsmodels.stats.diagnostic import het_arch

# 1. Calculamos los Residuos Estandarizados (z) dividiendo los errores entre su volatilidad real
# Ojo: Usamos las dos columnas del DataFrame 'df_volatilidad_real' que acabas de comprobar
df_volatilidad_real['residuos_estandarizados'] = (
    df_volatilidad_real['residuos_reales_usd'] / df_volatilidad_real['volatilidad_garch_usd']
)

# 2. Ejecutamos el Test de ARCH de Engle de control de calidad sobre los residuos estandarizados
lm_stat_z, p_value_z, f_stat_z, f_p_value_z = het_arch(
    df_volatilidad_real['residuos_estandarizados'], maxlag=5
)

print("============ AUDITORÍA DE CALIDAD INTERNA DE TU GARCH ============")
print(f"Estadístico LM de los residuos estandarizados: {lm_stat_z:.4f}")
print(f"p-valor de control (Debe ser > 0.05):           {p_value_z:.6f}")
print("==================================================================")

# Regla de decisión automatizada para el control de calidad
if p_value_z > 0.05:
    print(f"🟢 ¡CONTROL DE CALIDAD APROBADO! p-valor ({p_value_z:.4f}) > 0.05.")
    print("El modelo GARCH absorbió y limpió con éxito toda la aglomeración de volatilidad.")
    print("Los residuos estandarizados son homocedásticos. ¡Listos para calcular el VaR corporativo!")
else:
    print("🔴 CONTROL DE CALIDAD RECHAZADO: El modelo dejó turbulencia sin explicar.")


============ AUDITORÍA DE CALIDAD INTERNA DE TU GARCH ============
Estadístico LM de los residuos estandarizados: 5.1826
p-valor de control (Debe ser > 0.05):           0.393999
🟢 ¡CONTROL DE CALIDAD APROBADO! p-valor (0.3940) > 0.05.
El modelo GARCH absorbió y limpió con éxito toda la aglomeración de volatilidad.
Los residuos estandarizados son homocedásticos. ¡Listos para calcular el VaR corporativo!


/var/folders/jw/23cy6wrx7lj00m30z9y8q01r0000gn/T/ipykernel_25763/2818085561.py:10: FutureWarning: the 'maxlag' keyword is deprecated, use 'nlags' instead.
  lm_stat_z, p_value_z, f_stat_z, f_p_value_z = het_arch(


Al correrle el Test de ARCH a ese resultado final (\(z_{t}\)), lo que estás haciendo es auditar el trabajo del GARCH [finance].Como el p-valor saltó de 0.000000 a 0.3940 (que es mucho mayor a 0.05), el test te confirma matemáticamente que la varianza condicional de ese nuevo vector ya es una constante perfecta [finance].Demostraste que ya no quedan racimos de volatilidad escondidos, que los signos ya no se agrupan en el tiempo y que el residuo estandarizado se convirtió en un ruido blanco estocástico puro (i.i.d.) [finance].🎯 La conclusión para tu portafolio:Le aplicas el test para probar que tu modelo ya terminó su trabajo [finance]. Si el p-valor hubiera dado menor a 0.05, significaría que tu GARCH estuvo mal calibrado porque dejó pasar turbulencia sin explicar [finance]. Pero tu 0.3940 es la certificación de que tu motor GARCH(1,1) absorbió el 100% de la dinámica temporal del riesgo, dejando el sistema pacificado y listo para calcular el dinero real